# Demo — Try the Fine-tuned Model on Any Dataset

Point this notebook at a **live Socrata dataset** (a dataset **id** + its **domain**) and it will:

1. **Fetch** the dataset's schema — name, every column's type, and sample values — straight from the open-data portal.
2. **Generate** a **dataset description** and a **description for every column**, twice:
   - **BASE** — the raw, *un*-fine-tuned `Qwen/Qwen3-8B`
   - **FINE-TUNED** — the same model with the SFT+DPO adapter from `finetune_descriptions.ipynb`
3. **Show them side by side** so you can compare, with the dataset's existing (gold) text shown for reference when it has any.

The base model is loaded **once**; the adapter is flipped **on** for the fine-tuned pass and **off** (`model.disable_adapter()`) for the base pass — no second model load. This reuses the exact prompt format and task definitions from the finetune / evaluate notebooks, so what you see matches how the model was trained.

> **Where this runs:** generation needs the GPU (your Tillicum H200 / a Colab GPU). The fetch + display cells are pure-Python. Inputs are a Socrata **dataset id** (e.g. `f6w7-q2d2`) and a **domain** (e.g. `data.wa.gov`).

## 0. Install (GPU host only)

In [ ]:
%pip install -q --upgrade "transformers>=4.51" "peft>=0.13" "bitsandbytes>=0.45" "datasets>=2.20" "accelerate>=0.34" pandas ipywidgets

## 1. Configuration

In [ ]:
# @dryrun
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-8B"  # base model (un-fine-tuned baseline)
ADAPTER_PATH = (
    "adapters/qwen3-8b-desc-dpo"  # SFT+DPO adapter from finetune_descriptions.ipynb
)

# ── Default demo input (override per run via run_demo(uid, domain) or the form below) ──
DATASET_UID = "f6w7-q2d2"  # e.g. WA Electric Vehicle Population Data
DOMAIN = "data.wa.gov"  # the Socrata portal that hosts that dataset

# generation budgets (same as the evaluation notebook)
MAX_NEW_TOKENS = {"dataset_description": 256, "column_description": 64}

# describe at most this many columns (None = every column in the dataset)
MAX_COLUMNS_TO_DESCRIBE = None

print(f"Model   : {MODEL_ID}")
print(f"Adapter : {ADAPTER_PATH}")
print(f"Default : {DATASET_UID} @ {DOMAIN}")

In [ ]:
# --- Colab setup: re-root the adapter path to your Drive project folder (no-op off Colab) ---
from pathlib import Path

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB  # set False to use Colab's local /content disk instead of Drive
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        "/content/drive/MyDrive/MetadataLearn"
    )  # same folder used for training
else:
    PROJECT_DIR = Path(".")

# the adapter produced by finetune_descriptions.ipynb lives under PROJECT_DIR
ADAPTER_PATH = str(PROJECT_DIR / "adapters" / "qwen3-8b-desc-dpo")
print("Adapter :", ADAPTER_PATH)

## 2. Task definitions (identical to the finetune / evaluate notebooks)

Same `SYSTEM_PROMPT` and prompt wording the model was trained on, so the demo prompts match the training distribution.

In [ ]:
# @dryrun  — shared task definitions (identical in finetune + evaluate notebooks)
import json, random

SYSTEM_PROMPT = (
    "You are a precise technical writer for open-data catalogs. "
    "Write grounded, concise documentation using only the provided metadata. "
    "Do not speculate or invent statistics. Output only the requested text, "
    "with no preamble, headers, or markdown."
)

MIN_DATASET_DESC_CHARS = (
    40  # skip datasets whose gold description is too thin to learn from
)
MIN_COLUMN_DESC_CHARS = 10  # skip columns with no real description
MAX_SAMPLES_DATASET = (
    4  # sample values shown per column in the dataset-description prompt
)
MAX_SAMPLES_COLUMN = 8  # sample values shown in the column-description prompt
MAX_COLS_IN_PROMPT = 40  # cap columns listed in the dataset-description prompt


def _fmt_samples(samples, n):
    vals = [str(s) for s in (samples or [])[:n] if str(s).strip()]
    return ", ".join(vals) if vals else "(no samples)"


def _column_block(columns):
    lines = []
    for c in columns[:MAX_COLS_IN_PROMPT]:
        lines.append(
            f"- {c.get('name','')} ({c.get('type','')}): e.g. "
            f"{_fmt_samples(c.get('samples'), MAX_SAMPLES_DATASET)}"
        )
    if len(columns) > MAX_COLS_IN_PROMPT:
        lines.append(f"- ... (+{len(columns) - MAX_COLS_IN_PROMPT} more columns)")
    return "\n".join(lines)


def build_dataset_desc_example(rec):
    """One dataset-description training example, or None if the dataset is unusable."""
    desc = (rec.get("description") or "").strip()
    cols = rec.get("columns") or []
    if len(desc) < MIN_DATASET_DESC_CHARS or len(cols) < 2:
        return None
    user = (
        "Task: Write a concise description (2-4 sentences) for the following dataset, "
        "suitable for an open-data catalog. Use only the provided schema and sample values.\n\n"
        f"Dataset name: {rec.get('name','')}\n"
        f"Columns:\n{_column_block(cols)}\n\n"
        "Description:"
    )
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    return {
        "uid": rec.get("uid"),
        "task": "dataset_description",
        "column": None,
        "prompt_messages": prompt_messages,
        "messages": prompt_messages + [{"role": "assistant", "content": desc}],
        "target": desc,
    }


def build_column_desc_examples(rec):
    """All column-description training examples for one dataset."""
    out = []
    name = rec.get("name", "")
    ddesc = (rec.get("description") or "").strip()
    ddesc_short = (
        (ddesc[:300] + "...") if len(ddesc) > 300 else (ddesc or "(none provided)")
    )
    for c in rec.get("columns") or []:
        cdesc = (c.get("description") or "").strip()
        if len(cdesc) < MIN_COLUMN_DESC_CHARS:
            continue
        user = (
            "Task: Write a one-sentence description of the column named below, for the given "
            "dataset. Use only the provided context and sample values.\n\n"
            f"Dataset: {name}\n"
            f"Dataset description: {ddesc_short}\n"
            f"Column name: {c.get('name','')}\n"
            f"Column type: {c.get('type','')}\n"
            f"Sample values: {_fmt_samples(c.get('samples'), MAX_SAMPLES_COLUMN)}\n\n"
            "Column description:"
        )
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        out.append(
            {
                "uid": rec.get("uid"),
                "task": "column_description",
                "column": c.get("name"),
                "prompt_messages": prompt_messages,
                "messages": prompt_messages + [{"role": "assistant", "content": cdesc}],
                "target": cdesc,
            }
        )
    return out

## 3. Fetch a live dataset by id + domain (Socrata)

Reuses the fetch helpers from `build_metadata_store.ipynb` to pull the schema + sample values for one dataset on demand, then builds a generation prompt for the dataset **and every column**. (The training builder skips columns with no gold text — the demo describes them all, since generating those is the whole point.)

In [ ]:
# Socrata fetch helpers (from build_metadata_store.ipynb) — pull one dataset on demand.
import json, re, urllib.parse, urllib.request, urllib.error
from datetime import datetime, timezone

APP_TOKEN = ""  # optional Socrata app token (raises rate limits)
SAMPLES_PER_COLUMN = (
    8  # distinct sample values to keep per column (shown in the prompt)
)
ROWS_TO_SCAN = 100  # rows pulled to harvest those samples
REQUEST_TIMEOUT = 60  # seconds per HTTP request
SKIP_COMPUTED = True  # drop Socrata system/computed columns (fieldName starts with ":")


def _request(url):
    req = urllib.request.Request(url)
    if APP_TOKEN:
        req.add_header("X-App-Token", APP_TOKEN)
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
        return json.load(resp)


def strip_html(s):
    if not s:
        return ""
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def is_real_column(col):
    field = col.get("fieldName") or ""
    if not field:
        return False
    if SKIP_COMPUTED and field.startswith(":"):
        return False
    return True


def _sample_value(v):
    if isinstance(v, (dict, list)):
        return json.dumps(v, ensure_ascii=False, default=str)
    return v


def collect_samples(field, rows, cached_top, k):
    samples, seen = [], set()

    def add(raw):
        if raw is None or raw == "":
            return False
        val = _sample_value(raw)
        key = val if isinstance(val, str) else json.dumps(val, default=str)
        if key in seen:
            return False
        seen.add(key)
        samples.append(val)
        return True

    for r in rows:
        if field in r and add(r[field]) and len(samples) >= k:
            return samples
    for t in cached_top or []:  # fallback when rows didn't yield enough
        if add(t.get("item")) and len(samples) >= k:
            break
    return samples


def fetch_view(uid, domain):
    return _request(f"https://{domain}/api/views/{uid}.json")


def fetch_rows(uid, domain, limit):
    return _request(f"https://{domain}/resource/{uid}.json?$limit={limit}")


def fetch_row_count(uid, domain):
    """Best-effort exact row count via a SoQL count(*); None if unavailable."""
    try:
        data = _request(f"https://{domain}/resource/{uid}.json?$select=count(*)")
        if isinstance(data, list) and data:
            return int(next(iter(data[0].values())))
    except Exception:
        return None
    return None


def _license_name(view):
    lic = view.get("license")
    if isinstance(lic, dict):
        return (lic.get("name") or "").strip()
    if isinstance(lic, str):
        return lic.strip()
    return ""


def _find_update_frequency(view):
    """Best-effort lookup of an update/refresh frequency in Socrata metadata."""
    md = view.get("metadata") or {}
    cf = md.get("custom_fields") or {}
    for section in cf.values():
        if isinstance(section, dict):
            for k, v in section.items():
                if "frequency" in str(k).lower() and v:
                    return str(v).strip()
    val = md.get("accrualPeriodicity") or view.get("accrualPeriodicity")
    return str(val).strip() if val else ""


def build_record(uid, domain):
    """Fetch + assemble one dataset record: name, description, columns (type, samples, gold desc)."""
    view = fetch_view(uid, domain)
    try:
        rows = fetch_rows(uid, domain, ROWS_TO_SCAN)
    except Exception as e:  # keep going on cached samples if the rows endpoint fails
        rows = []
        print(f"  (row fetch failed: {type(e).__name__}: {e} — using cached samples)")

    columns = []
    for col in view.get("columns", []) or []:
        if not is_real_column(col):
            continue
        field = col.get("fieldName")
        cached_top = (col.get("cachedContents") or {}).get("top")
        columns.append(
            {
                "name": col.get("name"),
                "field": field,
                "type": col.get("dataTypeName"),
                "description": strip_html(col.get("description")),
                "samples": collect_samples(field, rows, cached_top, SAMPLES_PER_COLUMN),
            }
        )

    return {
        "uid": uid,
        "name": view.get("name") or uid,
        "description": strip_html(view.get("description")),
        "domain": domain,
        "page_url": f"https://{domain}/d/{uid}",
        "column_count": len(columns),
        "columns": columns,
        # ── source metadata for the Markdown card's "Quick Facts" block ──
        "meta": {
            "publisher": (view.get("attribution") or "").strip(),
            "category": (view.get("category") or "").strip(),
            "tags": view.get("tags") or [],
            "created_at": view.get("createdAt"),
            "updated_at": view.get("rowsUpdatedAt") or view.get("viewLastModified"),
            "license": _license_name(view),
            "update_frequency": _find_update_frequency(view),
            "row_count": fetch_row_count(uid, domain),
        },
    }


# ── Demo prompt builders: identical wording to training, but describe EVERY column ──
def _dataset_desc_user(rec):
    cols = rec.get("columns") or []
    return (
        "Task: Write a concise description (2-4 sentences) for the following dataset, "
        "suitable for an open-data catalog. Use only the provided schema and sample values.\n\n"
        f"Dataset name: {rec.get('name','')}\n"
        f"Columns:\n{_column_block(cols)}\n\n"
        "Description:"
    )


def _column_desc_user(rec, col):
    name = rec.get("name", "")
    ddesc = (rec.get("description") or "").strip()
    ddesc_short = (
        (ddesc[:300] + "...") if len(ddesc) > 300 else (ddesc or "(none provided)")
    )
    return (
        "Task: Write a one-sentence description of the column named below, for the given "
        "dataset. Use only the provided context and sample values.\n\n"
        f"Dataset: {name}\n"
        f"Dataset description: {ddesc_short}\n"
        f"Column name: {col.get('name','')}\n"
        f"Column type: {col.get('type','')}\n"
        f"Sample values: {_fmt_samples(col.get('samples'), MAX_SAMPLES_COLUMN)}\n\n"
        "Column description:"
    )


def build_demo_items(rec, max_columns=None):
    """One generation item for the dataset + one per column (all columns; gold text optional)."""
    items = [
        {
            "kind": "dataset_description",
            "label": rec.get("name", ""),
            "type": None,
            "samples": None,
            "gold": (rec.get("description") or "").strip(),
            "prompt_messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": _dataset_desc_user(rec)},
            ],
        }
    ]
    cols = rec.get("columns") or []
    if max_columns:
        cols = cols[:max_columns]
    for c in cols:
        items.append(
            {
                "kind": "column_description",
                "label": c.get("name", ""),
                "type": c.get("type"),
                "samples": c.get("samples") or [],
                "gold": (c.get("description") or "").strip(),
                "prompt_messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": _column_desc_user(rec, c)},
                ],
            }
        )
    return items

## 4. Load base model + adapter (GPU)

The base model is loaded once. If the adapter is present it's attached now; both passes share this single model — adapter **ON** = fine-tuned, **OFF** = base. If no adapter is found, only the base model is shown.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True,  # allow CPU offload for modules that don't fit in GPU
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

adapter_exists = bool(ADAPTER_PATH) and Path(ADAPTER_PATH).exists()
if adapter_exists:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    has_adapter = hasattr(model, "disable_adapter")
    print("Loaded adapter:", ADAPTER_PATH)
else:
    has_adapter = False
    print(
        f"⚠ No adapter at {ADAPTER_PATH} — only the BASE model will be shown "
        "(train one with finetune_descriptions.ipynb)."
    )
model.eval()

## 5. Generation helpers

In [ ]:
import re, time, contextlib

_THINK_RE = re.compile(
    r"^\s*<think>.*?</think>\s*", re.DOTALL
)  # drop any leading think block


def generate(prompt_messages, max_new_tokens):
    # enable_thinking=False matches the training-time render (empty think block in the prompt)
    try:
        text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:  # tokenizers without the enable_thinking kwarg
        text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()
    return _THINK_RE.sub("", decoded).strip()


def run_pass(items, label, adapter_enabled):
    """Generate for every item under one model config (adapter on = fine-tuned, off = base)."""
    if adapter_enabled or not has_adapter:
        ctx = contextlib.nullcontext()
    else:
        ctx = model.disable_adapter()  # turn the adapter OFF for the base pass
    preds, start = [], time.time()
    print(f"  {label}: generating {len(items)} descriptions ...")
    with ctx:
        for i, it in enumerate(items, 1):
            preds.append(generate(it["prompt_messages"], MAX_NEW_TOKENS[it["kind"]]))
            if i % 10 == 0 or i == len(items):
                rate = i / max(1e-9, time.time() - start)
                print(f"    [{i}/{len(items)}] ({rate:.1f} items/s)")
    return preds

## 6. Run the demo — side-by-side comparison

`run_demo(uid, domain)` fetches the dataset, generates the dataset + per-column descriptions with the **base** model and again with the **fine-tuned** adapter, and renders them side by side (with any existing *gold* text shown for reference). The cell runs the configured default; edit the call or use the form in §7 to try your own.

In [ ]:
import html as _html
import re as _re
import datetime as _dt
from pathlib import Path as _Path
from IPython.display import HTML, Markdown, display


def _fmt(text):
    out = _html.escape(text or "").replace("\n", "<br>")
    return out or "<em style='color:#999'>(empty)</em>"


def _samples_str(samples, n=6):
    vals = [str(s) for s in (samples or [])[:n] if str(s).strip()]
    return ", ".join(vals) if vals else "—"


_BASE_BG, _FT_BG, _GOLD_BG = "#fff3e0", "#e8f5e9", "#f5f5f5"
_BASE_HDR, _FT_HDR = "BASE (un-fine-tuned)", "FINE-TUNED (SFT+DPO)"
_FONT = "font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif"


def render_comparison(rec, items, base, ft):
    show_ft = ft is not None
    ds_base = base[0]
    ds_ft = ft[0] if show_ft else None
    ds_gold = items[0]["gold"]

    head = (
        f"<div style='{_FONT};border:1px solid #e2e2e2;border-radius:10px;"
        "padding:16px 20px;margin-bottom:14px;background:#fafafa'>"
        f"<div style='font-size:20px;font-weight:700'>{_html.escape(rec.get('name',''))}</div>"
        "<div style='color:#666;font-size:13px;margin-top:4px'>"
        f"<code>{_html.escape(rec.get('uid',''))}</code> &middot; {_html.escape(rec.get('domain',''))} "
        f"&middot; {len(rec.get('columns') or [])} columns &middot; "
        f"<a href='{_html.escape(rec.get('page_url',''))}' target='_blank'>portal &#8599;</a>"
        "</div></div>"
    )

    # ---- dataset description: base vs fine-tuned ----
    ds_cells = (
        f"<td style='width:50%;vertical-align:top;padding:10px;background:{_BASE_BG}'>"
        f"<div style='font-weight:700;font-size:12px;color:#b26a00;margin-bottom:6px'>{_BASE_HDR}</div>"
        f"{_fmt(ds_base)}</td>"
    )
    if show_ft:
        ds_cells += (
            f"<td style='width:50%;vertical-align:top;padding:10px;background:{_FT_BG}'>"
            f"<div style='font-weight:700;font-size:12px;color:#2e7d32;margin-bottom:6px'>{_FT_HDR}</div>"
            f"{_fmt(ds_ft)}</td>"
        )
    ds_block = (
        f"<div style='{_FONT}'>"
        "<div style='font-weight:700;font-size:15px;margin:6px 0'>Dataset description</div>"
        "<table style='width:100%;border-collapse:collapse;font-size:13px;line-height:1.5'>"
        f"<tr>{ds_cells}</tr></table>"
    )
    if ds_gold:
        ds_block += (
            f"<div style='margin-top:6px;padding:8px 10px;background:{_GOLD_BG};border-radius:6px;"
            f"font-size:12px;color:#555'><b>Existing (gold):</b> {_fmt(ds_gold)}</div>"
        )
    ds_block += "</div>"

    # ---- per-column table ----
    col_items = [
        (it, base[i], (ft[i] if show_ft else None))
        for i, it in enumerate(items)
        if it["kind"] == "column_description"
    ]
    rows_html = ""
    for it, b, f in col_items:
        gold_html = (
            f"<div style='margin-top:5px;font-size:11px;color:#777'><b>gold:</b> {_fmt(it['gold'])}</div>"
            if it["gold"]
            else ""
        )
        meta = (
            "<td style='vertical-align:top;padding:8px;width:18%'>"
            f"<div style='font-weight:600'>{_html.escape(it['label'] or '')}</div>"
            f"<div style='color:#888;font-size:11px'>{_html.escape(str(it['type'] or ''))}</div>"
            f"<div style='color:#aaa;font-size:11px;margin-top:3px'>e.g. {_html.escape(_samples_str(it['samples']))}</div></td>"
        )
        base_cell = (
            f"<td style='vertical-align:top;padding:8px;background:{_BASE_BG}'>"
            f"{_fmt(b)}{'' if show_ft else gold_html}</td>"
        )
        row = "<tr style='border-top:1px solid #eee'>" + meta + base_cell
        if show_ft:
            row += f"<td style='vertical-align:top;padding:8px;background:{_FT_BG}'>{_fmt(f)}{gold_html}</td>"
        rows_html += row + "</tr>"

    header = (
        "<th style='text-align:left;padding:8px'>Column</th>"
        f"<th style='text-align:left;padding:8px;color:#b26a00'>{_BASE_HDR}</th>"
    )
    if show_ft:
        header += (
            f"<th style='text-align:left;padding:8px;color:#2e7d32'>{_FT_HDR}</th>"
        )
    cols_table = (
        f"<div style='{_FONT};margin-top:18px'>"
        "<div style='font-weight:700;font-size:15px;margin:6px 0'>Column descriptions</div>"
        "<table style='width:100%;border-collapse:collapse;font-size:13px;line-height:1.45'>"
        f"<tr style='background:#fafafa'>{header}</tr>{rows_html}</table></div>"
    )

    display(HTML(head + ds_block + cols_table))


# ── Markdown "dataset card" export (same layout as 229f-7afc.md / 03_generate_cards) ──
_TYPE_LABELS = {
    "number": "Number",
    "text": "Text",
    "calendar_date": "Date",
    "checkbox": "Checkbox",
    "url": "URL",
    "money": "Money",
    "percent": "Percent",
    "point": "Point",
    "location": "Location",
    "multipoint": "MultiPoint",
    "multipolygon": "MultiPolygon",
    "multiline": "MultiLine",
    "line": "Line",
    "polygon": "Polygon",
    "document": "Document",
    "photo": "Photo",
}

_NOT_PROVIDED = "Not provided in source metadata."


def _pretty_type(t):
    t = (t or "").strip()
    return _TYPE_LABELS.get(t.lower(), t.replace("_", " ").title()) if t else ""


def _md_cell(text):
    """Flatten a value so it stays inside a single Markdown table cell."""
    s = _re.sub(r"\s*\n\s*", " ", (text or "").strip())
    return s.replace("\\", "\\\\").replace("|", "\\|") or "—"


def _md_inline(text):
    """Collapse a value onto one line for inline Markdown (list items, not table cells).

    Pipes are left as-is: the Quick Facts block is a bulleted list, so the ` | `
    separators in fields like size and dates render literally.
    """
    return _re.sub(r"\s*\n\s*", " ", (text or "").strip())


def _fmt_card_date(ts):
    """Format a Socrata unix timestamp as 'January 22, 2026' (or None)."""
    try:
        dt = _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc)
    except (TypeError, ValueError, OSError, OverflowError):
        return None
    return f"{dt.strftime('%B')} {dt.day}, {dt.year}"


def _quick_facts_lines(rec):
    """Bulleted 'Quick Facts' block built from the dataset's source metadata."""
    meta = rec.get("meta") or {}

    publisher = (meta.get("publisher") or "").strip() or _NOT_PROVIDED

    parts = []
    category = (meta.get("category") or "").strip()
    if category:
        parts.append(category.lower())
    parts.extend(t.strip() for t in (meta.get("tags") or []) if t and t.strip())
    category_tags = " | ".join(parts) if parts else _NOT_PROVIDED

    frequency = (meta.get("update_frequency") or "").strip() or _NOT_PROVIDED

    n_cols = len(rec.get("columns") or [])
    n_rows = meta.get("row_count")
    rows_str = (
        f"{n_rows:,} Rows" if isinstance(n_rows, int) and n_rows >= 0 else _NOT_PROVIDED
    )
    size = f"{rows_str} | {n_cols} Columns"

    created = _fmt_card_date(meta.get("created_at"))
    updated = _fmt_card_date(meta.get("updated_at"))
    date_bits = []
    if created:
        date_bits.append(f"Created {created}")
    if updated:
        date_bits.append(f"Last Updated {updated}")
    dates = " | ".join(date_bits) if date_bits else _NOT_PROVIDED

    license_name = (meta.get("license") or "").strip() or _NOT_PROVIDED

    facts = [
        ("Publisher", publisher),
        ("Category / Tags", category_tags),
        ("Update Frequency", frequency),
        ("Dataset Size", size),
        ("Dates", dates),
        ("License", license_name),
    ]
    return [f"- **{label}:** {_md_inline(value)}" for label, value in facts]


def _context_block(rec):
    """Code-built 'Context, Provenance, & Limitations' section (no LLM; see 03_generate_cards).

    Everything here is extracted deterministically from the payload — zero
    hallucination risk: a provenance pointer to the source endpoint, the
    publisher's own description (if any), and fixed reliability/schema caveats.
    """
    uid = (rec.get("uid") or "").strip()
    domain = (rec.get("domain") or "").strip()
    source_note = (
        f"Source metadata obtained from https://{domain}/resource/{uid}.json."
        if uid and domain
        else _NOT_PROVIDED
    )

    publisher_desc = _md_inline(rec.get("description") or "")
    global_note = (
        f"\n\n**Publisher Description:** {publisher_desc}"
        if publisher_desc and publisher_desc != _NOT_PROVIDED
        else ""
    )

    return f"""\
## Context, Provenance, & Limitations

**Data Source:** {source_note}{global_note}

**Data Reliability:** Best-effort compilation of source records. \\
Users should verify against original source documents where precision is required.

**Schema Note:** Column count reflects columns present at time of extraction. \\
Dataset schema may evolve; consult the source portal for current field definitions."""


def dataset_card_markdown(rec, items, base, ft, prefer="ft"):
    """Render a run_demo() result as a Markdown dataset card (see 229f-7afc.md).

    Layout (same order as 03_generate_cards): title, code-built Quick Facts,
    LLM Dataset Overview, LLM Data Dictionary, code-built Context/Provenance.
    prefer="ft" uses the fine-tuned text (falls back to base if no adapter);
    prefer="base" forces the un-fine-tuned baseline.
    """
    preds = ft if (prefer == "ft" and ft is not None) else base
    name = rec.get("name") or rec.get("uid", "")
    lines = [
        f"# Dataset Card: {name}",
        "",
        "## Quick Facts",
        "",
        *_quick_facts_lines(rec),
        "",
        "## Dataset Overview",
        "",
        (preds[0] or "").strip() or "_(no description generated)_",
        "",
        "## Data Dictionary",
        "",
        "| Column Name | Type | Description |",
        "| :--- | :--- | :--- |",
    ]
    for i, it in enumerate(items):
        if it["kind"] != "column_description":
            continue
        lines.append(
            f"| **{_md_cell(it['label'])}** | `{_pretty_type(it['type'])}` | {_md_cell(preds[i])} |"
        )
    lines += ["", _context_block(rec)]
    return "\n".join(lines) + "\n"


def save_dataset_card(rec, items, base, ft, prefer="ft", out_dir=None, show=True):
    """Write the dataset card to <out_dir>/<uid>.md and (optionally) render it inline."""
    md = dataset_card_markdown(rec, items, base, ft, prefer=prefer)
    base_dir = _Path(out_dir) if out_dir is not None else _Path(".")
    out_path = base_dir / f"{rec.get('uid', 'dataset')}.md"
    out_path.write_text(md, encoding="utf-8")
    src = "fine-tuned" if (prefer == "ft" and ft is not None) else "base"
    print(f"Wrote {out_path}  ({src} descriptions)")
    if show:
        display(Markdown(md))
    return md


def run_demo(
    uid=None,
    domain=None,
    max_columns=MAX_COLUMNS_TO_DESCRIBE,
    save_md=True,
    prefer="ft",
):
    """Fetch a dataset, generate base + fine-tuned descriptions, show them side by side.

    With save_md=True (default) it also writes a Markdown dataset card to
    PROJECT_DIR/<uid>.md and renders it below the comparison. The full result is
    cached in the global LAST_DEMO so you can re-export without regenerating.
    """
    global LAST_DEMO
    uid = (uid or DATASET_UID).strip()
    domain = (domain or DOMAIN).strip()
    print(f"Fetching {uid} from {domain} ...")
    rec = build_record(uid, domain)
    items = build_demo_items(rec, max_columns=max_columns)
    print(f"  {rec['name']} — describing 1 dataset + {len(items) - 1} columns\n")
    base = run_pass(items, "BASE", adapter_enabled=False)
    ft = run_pass(items, "FINE-TUNED", adapter_enabled=True) if has_adapter else None
    render_comparison(rec, items, base, ft)
    LAST_DEMO = (rec, items, base, ft)
    if save_md:
        print()
        save_dataset_card(rec, items, base, ft, prefer=prefer, out_dir=PROJECT_DIR)
    return rec, items, base, ft


# Run on the configured default — or call run_demo("<uid>", "<domain>") with your own.
_ = run_demo(DATASET_UID, DOMAIN)

## 7. (Optional) Interactive form

Type a dataset id + domain and click **Generate & compare**. Falls back to a printed hint if `ipywidgets` isn't available — in that case just call `run_demo("<uid>", "<domain>")` directly.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    _sty = {"description_width": "initial"}
    uid_box = widgets.Text(value=DATASET_UID, description="Dataset id:", style=_sty)
    dom_box = widgets.Text(value=DOMAIN, description="Domain:", style=_sty)
    cap_box = widgets.IntText(
        value=(MAX_COLUMNS_TO_DESCRIBE or 0),
        description="Max cols (0 = all):",
        style=_sty,
    )
    go = widgets.Button(description="Generate & compare", button_style="primary")
    out = widgets.Output()

    def _on_click(_):
        with out:
            clear_output()
            run_demo(uid_box.value, dom_box.value, max_columns=(cap_box.value or None))

    go.on_click(_on_click)
    display(widgets.VBox([uid_box, dom_box, cap_box, go, out]))
except Exception as e:
    print("ipywidgets unavailable —", e)
    print('Call it directly instead, e.g.:  run_demo("f6w7-q2d2", "data.wa.gov")')

## 8. Export as a Markdown dataset card

`run_demo(...)` already writes a Markdown **dataset card** to `PROJECT_DIR/<uid>.md` after each run (set `save_md=False` to skip it) — same layout as `229f-7afc.md` / `03_generate_cards`:

1. `# Dataset Card` title
2. `## Quick Facts` — code-built from source metadata (publisher, category/tags, update frequency, size, dates, license); shows `Not provided in source metadata.` for anything the portal doesn't expose.
3. `## Dataset Overview` — the generated paragraph.
4. `## Data Dictionary` — generated one-line description per column.
5. `## Context, Provenance, & Limitations` — code-built (zero hallucination risk): a provenance pointer to the source `…/resource/<uid>.json` endpoint, the publisher's own description when present, and fixed reliability/schema caveats.

The descriptions in 3–4 use the **fine-tuned** text by default (falling back to the base model if no adapter is loaded); 2 and 5 are extracted deterministically from the fetched payload. The most recent run is cached in `LAST_DEMO`, so you can re-render or re-export below **without** generating again.

In [ ]:
# Re-export the most recent run as Markdown (no regeneration — reuses LAST_DEMO).
rec, items, base, ft = LAST_DEMO
md = save_dataset_card(rec, items, base, ft, prefer="ft", out_dir=PROJECT_DIR)

# Other options (all reuse the cached run):
#   save_dataset_card(*LAST_DEMO, prefer="base")          # un-fine-tuned baseline card
#   save_dataset_card(*LAST_DEMO, out_dir="cards")        # write to cards/<uid>.md
#   print(dataset_card_markdown(*LAST_DEMO))              # just the string, no file